<a href="https://colab.research.google.com/github/tettty1981-lang/data-analytics-course/blob/main/HW_16_2_%D0%9F%D1%80%D0%BE%D0%B3%D0%BD%D0%BE%D0%B7%D1%83%D0%B2%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BA%D1%80%D0%B5%D0%B4%D0%B8%D1%82%D0%BD%D0%BE%D0%B3%D0%BE_%D1%80%D0%B8%D0%B7%D0%B8%D0%BA%D1%83.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Прогнозування кредитного ризику

## Мета завдання
Застосувати знання з лекції для побудови моделі логістичної регресії, що прогнозує ймовірність дефолту за кредитом, іншими словами, що людина не поверне кредит. Ви пройдете весь цикл: від дослідницького аналізу до оцінки якості класифікаційної моделі.

## Опис датасету
**Credit Risk Dataset** містить інформацію про 32,000+ позичальників з такими параметрами:
- **person_age**: Вік позичальника
- **person_income**: Річний дохід
- **person_home_ownership**: Тип володіння житлом
- **person_emp_length**: Стаж роботи (в роках)
- **loan_intent**: Мета кредиту
- **loan_grade**: Кредитний рейтинг (A - кращий, F - гірший)
- **loan_amnt**: Сума кредиту
- **loan_int_rate**: Процентна ставка
- **loan_status**: Статус кредиту (0 = сплачено, 1 = дефолт) - **цільова змінна**
- **loan_percent_income**: Відношення кредиту до доходу
- **cb_person_default_on_file**: Історія дефолтів (Y/N)
- **cb_person_cred_hist_length**: Довжина кредитної історії

---

## Завдання 1: Завантаження та перший огляд даних (1 бал)

**Що потрібно зробити:**
1. Завантажте дані з файлу `credit_risk_dataset.csv`
2. Виведіть розмір датасету
3. Покажіть перші 5 рядків
4. Виведіть загальну інформацію про дані (кількість записів, типи колонок)
5. Перевірте розподіл цільової змінної (відсотк даних для кожного класу)

Дайте висновок, це задача збалансованої чи незбалансованої класифікації.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('credit_risk_dataset.csv')
print(f"Розмір датасету: {df.shape}")
print(f"Кількість клієнтів: {df.shape[0]:,}")
print(f"Кількість ознак: {df.shape[1]}")

Розмір датасету: (32581, 12)
Кількість клієнтів: 32,581
Кількість ознак: 12


In [3]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [6]:
df['loan_status'].value_counts(normalize=True) * 100

,proportion
loan_status,
0,78.183604
1,21.816396


**Висновок.** Серед позичальників 78,2% повернули кредит (клас 0) і 21,8% потрапили в дефолт (клас 1). Класи розподілені нерівномірно: дефолтів майже вчетверо менше, тож це задача незбалансованої класифікації. Через це самої accuracy для оцінки моделі недостатньо: модель, яка завжди передбачає «поверне», мала б ~78% точності, але не знаходила б жодного дефолту. Тому далі оцінюватимемо модель ще й за precision, recall, F1 та AUC.


---

## Завдання 2: Дослідницький аналіз даних (EDA) (4 бали)

**Що потрібно зробити:**
1. Обчисліть відсоток пропущених значень в колонках. За наявності пропущених значень - заповніть їх медіаною для числових колонок і найбільш частим значеннмя для категоріальних.
2. Проаналізуйте розподіл числових змінних.
3. Знайдіть та обробіть викиди в колонці person_income з допомогою [Interquartile range](https://uk.wikipedia.org/wiki/%D0%9C%D1%96%D0%B6%D0%BA%D0%B2%D0%B0%D1%80%D1%82%D0%B8%D0%BB%D1%8C%D0%BD%D0%B8%D0%B9_%D1%80%D0%BE%D0%B7%D0%BC%D0%B0%D1%85).
4. Проаналізуйте категоріальні змінні відносно цільової та частоту зустрічання різних значень в них.
5. Візуалізуйте взаємозв'язок ознак з цільовою змінною.


In [5]:
missing = df.isnull().sum()
missing[missing > 0]

,0
person_emp_length,895
loan_int_rate,3116


In [7]:
# Відсоток пропусків
print((df.isnull().mean() * 100).round(2)[df.isnull().any()])

# Заповнення медіаною
for col in ['person_emp_length', 'loan_int_rate']:
    df[col] = df[col].fillna(df[col].median())

# Перевірка
print(df.isnull().sum().sum())  # має бути 0

person_emp_length    2.75
loan_int_rate        9.56
dtype: float64
0


In [14]:
# Помилкові значення
df = df[(df['person_age'] <= 100) & (df['person_emp_length'] <= 60)]

num_cols = df.select_dtypes(include='number').columns.drop('loan_status')
df[num_cols].describe().round(2)

fig = make_subplots(rows=3, cols=3, subplot_titles=list(num_cols))
for i, col in enumerate(num_cols):
    fig.add_trace(go.Histogram(x=df[col], nbinsx=50, showlegend=False),
                  row=i // 3 + 1, col=i % 3 + 1)
fig.update_layout(height=800, title_text='Розподіл числових змінних')
fig.show()

In [15]:
Q1 = df['person_income'].quantile(0.25)
Q3 = df['person_income'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

n_out = ((df['person_income'] < lower) | (df['person_income'] > upper)).sum()
print(f"Межі: {lower:.0f} – {upper:.0f}; викидів: {n_out} ({n_out / len(df):.2%})")

df['person_income'] = df['person_income'].clip(lower, upper)

Межі: -22550 – 140250; викидів: 1480 (4.54%)


In [16]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    stats = df.groupby(col)['loan_status'].agg(count='size', default_rate='mean')
    stats['share_%'] = (stats['count'] / len(df) * 100).round(2)
    stats['default_rate'] = (stats['default_rate'] * 100).round(2)
    print(f"\n{col}")
    print(stats.sort_values('default_rate', ascending=False))


person_home_ownership
                       count  default_rate  share_%
person_home_ownership                              
RENT                   16442         31.57    50.48
OTHER                    107         30.84     0.33
MORTGAGE               13441         12.57    41.26
OWN                     2584          7.47     7.93

loan_intent
                   count  default_rate  share_%
loan_intent                                    
DEBTCONSOLIDATION   5212         28.59    16.00
MEDICAL             6071         26.70    18.64
HOMEIMPROVEMENT     3605         26.10    11.07
PERSONAL            5519         19.88    16.94
EDUCATION           6451         17.22    19.80
VENTURE             5716         14.82    17.55

loan_grade
            count  default_rate  share_%
loan_grade                              
G              64         98.44     0.20
F             241         70.54     0.74
E             964         64.42     2.96
D            3625         59.03    11.13
C         

**Частота значень.** Половина клієнтів орендує житло (50,5%), ще 41,3% мають іпотеку. Серед цілей кредиту значення розподілені рівномірно (11–20%). Клас кредитного рейтингу A–B охоплює 65% клієнтів, а класи E–G малочисельні (~4%). У 17,6% клієнтів є дефолт в історії.

**Зв'язок із дефолтом.** Найсильніше впливає loan_grade: частка дефолтів зростає від 9,96% (A) до 98,4% (G). Орендарі мають 31,6% дефолтів проти 7,5% у власників житла. Клієнти з минулим дефолтом дефолтять удвічі частіше (37,8% проти 18,4%). Серед цілей кредиту найризикованіші рефінансування боргів (28,6%) і медичні витрати (26,7%), найменш ризикований VENTURE (14,8%).

**Для порівняння**: загальна частка дефолтів у даних становить 21,8%, тому категорії з показником значно вищим за це значення (оренда житла, класи D–G, дефолт в історії) є групами підвищеного ризику.

In [17]:
# Числові: box plot за статусом кредиту
fig = make_subplots(rows=3, cols=3, subplot_titles=list(num_cols))
for i, col in enumerate(num_cols):
    for status, color in [(0, '#636EFA'), (1, '#EF553B')]:
        fig.add_trace(
            go.Box(y=df.loc[df['loan_status'] == status, col],
                   name=str(status), marker_color=color, showlegend=(i == 0)),
            row=i // 3 + 1, col=i % 3 + 1)
fig.update_layout(height=800, title_text='Числові ознаки за loan_status')
fig.show()

# Категоріальні: частка дефолтів за категорією
fig = make_subplots(rows=2, cols=2, subplot_titles=list(cat_cols))
for i, col in enumerate(cat_cols):
    rate = df.groupby(col)['loan_status'].mean().sort_values(ascending=False) * 100
    fig.add_trace(go.Bar(x=rate.index, y=rate.values, showlegend=False),
                  row=i // 2 + 1, col=i % 2 + 1)
fig.update_yaxes(title_text='% дефолтів')
fig.update_layout(height=700, title_text='Частка дефолтів за категоріями')
fig.show()

### Висновок: взаємозв'язок ознак із цільовою змінною

**Числові ознаки.** Найсильніше відрізняються класи за трьома ознаками:

- `loan_int_rate`: у клієнтів із дефолтом медіана процентної ставки помітно вища (~13% проти ~11%).
- `loan_percent_income`: частка кредиту від доходу в дефолтних клієнтів вища (медіана ~0,24 проти ~0,13), тобто більше навантаження на дохід.
- `person_income`: у дефолтних клієнтів медіана доходу нижча (~42 тис. проти ~60 тис.).

Слабший зв'язок у `loan_amnt` (у дефолтних сума трохи більша) і `person_emp_length` (стаж трохи менший). `person_age` і `cb_person_cred_hist_length` майже не розрізняють класи, їхні розподіли практично збігаються.

**Категоріальні ознаки.** Найвищий відсоток дефолтів у клієнтів із `loan_grade` D–G (59–98%), у орендарів (RENT,приблизно 32%) і в тих, у кого є дефолт в історії (Y, приблизно 38%). Найнижчий у власників житла (OWN,приблизно 7%) і з класом A (~10%). Цілі кредиту розрізняють класи помірніше: від ~15% (VENTURE) до ~29% (DEBTCONSOLIDATION).

**Загалом:** на ймовірність дефолту найбільше впливають кредитний клас, процентна ставка, відношення кредиту до доходу, дохід і тип житла. Вік і довжина кредитної історії малоінформативні.


---

## Завдання 3: Аналіз кореляцій та Feature Engineering (3 бали)

**Що потрібно зробити:**
1. Побудуйте матрицю кореляцій для числових змінних.
2. Закодуйте категоріальні змінні.
3. Виберіть фінальний набір ознак, можна лишити всі, якщо ви вважаєте, що це - доцільно.



In [18]:
num_cols = df.select_dtypes(include='number').columns.drop('loan_status')
corr = df[list(num_cols) + ['loan_status']].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Матриця кореляцій')
fig.update_layout(height=700, width=800)
fig.show()

# Кореляція з цільовою змінною
corr['loan_status'].drop('loan_status').sort_values(key=abs, ascending=False)

,loan_status
loan_percent_income,0.38
loan_int_rate,0.32
person_income,-0.25
loan_amnt,0.11
person_emp_length,-0.09
person_age,-0.02
cb_person_cred_hist_length,-0.02


### Висновок: аналіз кореляцій

**Зв'язок із цільовою змінною.** Найсильніше з `loan_status` корелюють:
- `loan_percent_income` (0,38) і `loan_int_rate` (0,32): чим більше навантаження кредиту на дохід і вища ставка, тим вищий ризик дефолту;
- `person_income` (−0,25): чим вищий дохід, тим менший ризик.

Слабкий зв'язок мають `loan_amnt` (0,11) і `person_emp_length` (−0,09). `person_age` і `cb_person_cred_hist_length` практично не пов'язані з ціллю (−0,02).

**Зв'язок між ознаками.** Дуже сильна кореляція між `person_age` і `cb_person_cred_hist_length` (0,88): довша кредитна історія майже завжди означає старший вік, тож ці ознаки дублюють одна одну (мультиколінеарність). Помірні кореляції: `loan_amnt` і `loan_percent_income` (0,57), `person_income` і `loan_amnt` (0,42), `person_income` і `loan_percent_income` (−0,37). Вони логічні, бо `loan_percent_income` розраховується з суми кредиту та доходу.

*Примітка: коефіцієнти Пірсона відображають лише лінійний зв'язок.*

In [21]:
# Пункт 2: кодування (без змін)
df_enc = df.copy()
df_enc['cb_person_default_on_file'] = df_enc['cb_person_default_on_file'].map({'Y': 1, 'N': 0})
grade_order = {g: i for i, g in enumerate(['A', 'B', 'C', 'D', 'E', 'F', 'G'])}
df_enc['loan_grade'] = df_enc['loan_grade'].map(grade_order)
df_enc = pd.get_dummies(df_enc, columns=['person_home_ownership', 'loan_intent'],
                        drop_first=True, dtype=int)

# Пункт 3: фінальний набір
target = 'loan_status'
features = [c for c in df_enc.columns if c not in [target, 'cb_person_cred_hist_length']]
print(len(features))
print(features)

16
['person_age', 'person_income', 'person_emp_length', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


### Висновок: кодування змінних і вибір ознак

**Кодування категоріальних змінних.**
- `cb_person_default_on_file`: бінарна ознака, Y → 1, N → 0.
- `loan_grade`: порядкова ознака, A → 0 … G → 6. Класи мають природний порядок, а частка дефолтів зростає разом із ним.
- `person_home_ownership` і `loan_intent`: номінальні ознаки без порядку, застосовано one-hot кодування з `drop_first=True`, щоб уникнути дублювання колонок.

**Фінальний набір ознак.** До моделі увійшло 16 ознак. Колонку `cb_person_cred_hist_length` виключено: вона сильно корелює з `person_age` (0,88), тобто дублює її, і майже не пов'язана з цільовою змінною (−0,02). Решту ознак залишено, бо їх небагато відносно розміру вибірки (32 574 рядки), а слабкі ознаки не зашкодять логістичній регресії з масштабуванням. Пари `loan_amnt` та `loan_percent_income` (0,57) і `person_income` та `loan_amnt` (0,42) мають помірну кореляцію, тому не вважаються проблемною мультиколінеарністю.


---

## Завдання 4: Підготовка даних та навчання моделі (3 бали)

**Що потрібно зробити:**
1. Розділіть дані на X та y
2. Поділіть на навчальну та тестову вибірки
3. Застосуйте масштабування
4. Навчіть модель логістичної регресії
5. Зробіть прогнози на тренувальній та тестовій вибірках.



In [22]:
X = df_enc[features]
y = df_enc['loan_status']
print(X.shape, y.shape)

(32574, 16) (32574,)


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))  # частка дефолтів в обох вибірках

(26059, 16) (6515, 16)
0.2182 0.2181


In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # навчаємо scaler тільки на train
X_test_scaled = scaler.transform(X_test)         # до test лише застосовуємо

In [25]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [26]:
# Класи (поріг 0.5)
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Ймовірності дефолту (знадобляться для ROC-кривої й порогів)
y_train_proba = model.predict_proba(X_train_scaled)[:, 1]
y_test_proba = model.predict_proba(X_test_scaled)[:, 1]

print(y_test_pred[:10])
print(y_test_proba[:10].round(3))

[0 1 0 1 0 0 0 0 0 0]
[0.045 0.826 0.107 0.843 0.03  0.215 0.017 0.016 0.023 0.005]


### Висновок: підготовка даних і навчання моделі

Дані розділено на ознаки (X, 16 колонок) і цільову змінну (y = `loan_status`), після чого поділено на навчальну (80%) і тестову (20%) вибірки з параметром `stratify`, щоб частка дефолтів (~21,8%) була однаковою в обох вибірках. Ознаки масштабовано `StandardScaler` (параметри обчислено лише на навчальній вибірці, щоб уникнути витоку даних). Навчено логістичну регресію і зроблено прогнози класів та ймовірностей дефолту на тренувальній і тестовій вибірках.


---

## Завдання 5: Оцінка якості моделі (4 бали)

**Що потрібно зробити:**
1. Побудуйте confusion matrix.
2. Обчисліть основні метрики (accuracy, precision, recall, f1).
3. Побудуйте ROC-криву та обчисліть AUC
4. Проаналізуйте важливість ознак.
5. Зробіть висновки про якість моделі та які ознаки найбільше впливають на прогноз.


In [27]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                x=['Передбачено 0', 'Передбачено 1'],
                y=['Фактично 0', 'Фактично 1'],
                title='Confusion matrix (тест, поріг 0.5)')
fig.show()

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = pd.DataFrame({
    'train': [accuracy_score(y_train, y_train_pred), precision_score(y_train, y_train_pred),
              recall_score(y_train, y_train_pred), f1_score(y_train, y_train_pred)],
    'test':  [accuracy_score(y_test, y_test_pred), precision_score(y_test, y_test_pred),
              recall_score(y_test, y_test_pred), f1_score(y_test, y_test_pred)],
}, index=['accuracy', 'precision', 'recall', 'f1']).round(3)
metrics

,train,test
accuracy,0.854,0.850
precision,0.740,0.731
recall,0.510,0.495
f1,0.604,0.591


In [29]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(y_test, y_test_proba)
auc = roc_auc_score(y_test, y_test_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f'Logistic Regression (AUC = {auc:.3f})'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name='Випадкове вгадування',
                         line=dict(dash='dash')))
fig.update_layout(title='ROC-крива', xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate')
fig.show()
print(f'AUC: {auc:.3f}')

AUC: 0.857


In [30]:
importance = pd.DataFrame({
    'feature': features,
    'coef': model.coef_[0]
}).sort_values('coef', key=abs, ascending=True)

fig = px.bar(importance, x='coef', y='feature', orientation='h',
             title='Коефіцієнти логістичної регресії (ознаки масштабовано)')
fig.update_layout(height=600)
fig.show()

In [31]:
importance.round(2)

,feature,coef
8,person_home_ownership_OTHER,0.01
0,person_age,-0.02
2,person_emp_length,-0.02
12,loan_intent_HOMEIMPROVEMENT,0.03
7,cb_person_default_on_file,-0.06
5,loan_int_rate,-0.06
13,loan_intent_MEDICAL,-0.08
1,person_income,-0.16
14,loan_intent_PERSONAL,-0.21
11,loan_intent_EDUCATION,-0.33


### Висновок: оцінка якості моделі

**Confusion matrix (тест, поріг 0.5).** З 6 515 клієнтів модель правильно визначила 4 835 сумлінних позичальників (TN) і 704 дефолти (TP). Помилилась у 259 випадках, віднісши надійних клієнтів до ризикових (FP), і у 717 випадках пропустила дефолт (FN).

**Метрики (тест).** accuracy = 0,850, precision = 0,731, recall = 0,495, F1 = 0,591. Метрики на тренувальній і тестовій вибірках майже збігаються (accuracy 0,854 і 0,850), тож перенавчання немає.

**ROC-крива.** AUC = 0,857: модель добре відрізняє дефолтних клієнтів від надійних (випадкове вгадування дало б 0,5).

**Загальна оцінка.** Модель якісна за AUC і precision: якщо вона прогнозує дефолт, то в 73% випадків має рацію. Проте recall лише 0,495, тобто при порозі 0,5 модель пропускає майже половину реальних дефолтів. Висока accuracy (0,85) частково пояснюється дисбалансом класів: модель, що завжди прогнозує «поверне», мала б accuracy ~0,78. Для кредитного ризику пропущений дефолт зазвичай коштує дорожче, ніж відмова надійному клієнтові, тому поріг варто знизити (завдання 6).

**Важливість ознак.** Найбільший вплив мають:
- `loan_percent_income` (коеф. +1,25) і `loan_grade` (+1,13): чим більше навантаження кредиту на дохід і гірший кредитний клас, тим вища ймовірність дефолту;
- тип житла: оренда (RENT, +0,38) підвищує ризик, власне житло (OWN, −0,43) знижує відносно іпотеки;
- ціль кредиту: VENTURE (−0,40) та EDUCATION (−0,33) знижують ризик відносно базової категорії DEBTCONSOLIDATION.

Від'ємні коефіцієнти в `loan_amnt` (−0,49) і `loan_int_rate` (−0,06) суперечать їхній кореляції з ціллю, але це наслідок взаємозалежності ознак: `loan_amnt` входить у `loan_percent_income`, а ставка значною мірою визначається `loan_grade`. Коли ці ознаки вже в моделі, `loan_amnt` і `loan_int_rate` майже не додають нової інформації. Тому коефіцієнти слід читати як вплив ознаки при фіксованих інших, а не як ізольований ефект.

Майже не впливають на прогноз `person_age` (−0,02), `person_emp_length` (−0,02), `loan_intent_HOMEIMPROVEMENT` (+0,03) і `person_home_ownership_OTHER` (+0,01).


---

## Завдання 6: Оптимізація порогу та бізнес-аналіз (2 бали)

**Що потрібно зробити:**
1. Проаналізуйте метрики precision, recall, F1 при різних порогах класифікації (мінімум - 5 різних порогів). Візуалізуйте як змінюються метрики якості при зміні порогу.
2. Оберіть оптимальний поріг для бізнес-задачі - можна обрати виходячи з попереднього пункту, або додати своїх роздумів і обрати інший.



In [32]:
thresholds = np.arange(0.1, 0.91, 0.05).round(2)

rows = []
for t in thresholds:
    pred = (y_test_proba >= t).astype(int)
    rows.append({
        'threshold': t,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'flagged_%': pred.mean() * 100,  # яку частку клієнтів позначаємо ризиковими
    })

th_df = pd.DataFrame(rows).round(3)
th_df

,threshold,precision,recall,f1,flagged_%
0,0.10,0.377,0.879,0.528,50.867
1,0.15,0.437,0.827,0.572,41.228
2,0.20,0.496,0.779,0.606,34.275
3,0.25,0.544,0.728,0.623,29.210
4,0.30,0.589,0.688,0.634,25.480
5,0.35,0.623,0.640,0.631,22.410
6,0.40,0.665,0.590,0.626,19.355
7,0.45,0.702,0.548,0.616,17.038
8,0.50,0.731,0.495,0.591,14.781
9,0.55,0.766,0.444,0.562,12.648


In [33]:
fig = go.Figure()
for m in ['precision', 'recall', 'f1']:
    fig.add_trace(go.Scatter(x=th_df['threshold'], y=th_df[m], mode='lines+markers', name=m))
fig.update_layout(title='Метрики якості залежно від порогу класифікації',
                  xaxis_title='Поріг', yaxis_title='Значення метрики')
fig.show()

In [34]:
best_f1 = th_df.loc[th_df['f1'].idxmax()]
print('Поріг з максимальним F1:')
print(best_f1)

# Бізнес-варіант: ловимо щонайменше 70% дефолтів, серед таких порогів беремо найвищий precision
candidates = th_df[th_df['recall'] >= 0.70]
business = candidates.loc[candidates['precision'].idxmax()]
print('\nПоріг для recall ≥ 0.70 з найкращим precision:')
print(business)


Поріг з максимальним F1:
threshold     0.300
precision     0.589
recall        0.688
f1            0.634
flagged_%    25.480
Name: 4, dtype: float64

Поріг для recall ≥ 0.70 з найкращим precision:
threshold     0.250
precision     0.544
recall        0.728
f1            0.623
flagged_%    29.210
Name: 3, dtype: float64


### Висновок: оптимізація порогу та бізнес-аналіз

**Зміна метрик залежно від порогу.** Досліджено 17 порогів від 0,10 до 0,90. Зі зростанням порогу precision зростає (з 0,377 при порозі 0,10 до 0,93 при 0,85–0,90), а recall падає (з 0,879 до 0,09): модель стає обережнішою, рідше хибно позначає клієнтів як ризикових, але пропускає більше реальних дефолтів. F1 має максимум 0,634 при порозі 0,30, далі спадає. Стандартний поріг 0,5 дає recall лише 0,495, тобто пропускає половину дефолтів.

**Вибір порогу.** Обираємо поріг **0,25**: precision = 0,544, recall = 0,728, F1 = 0,623. Він трохи поступається максимальному F1 (0,634 при порозі 0,30), але ловить на ~4 в.п. більше дефолтів (72,8% проти 68,8%).

Обґрунтування. Для кредитного ризику пропущений дефолт (FN) обходиться банку дорожче, ніж відмова надійному клієнтові (FP): у першому випадку втрачається тіло кредиту, у другому лише відсотковий дохід. Порівняно з порогом 0,5 поріг 0,25 ловить приблизно на 330 дефолтів більше (на тестовій вибірці з 6 515 клієнтів) ціною приблизно 610 додаткових хибних тривог. Тобто на кожен додатково виявлений дефолт припадає менше двох хибно відхилених надійних клієнтів. Якщо збиток від одного пропущеного дефолту перевищує збиток від двох втрачених надійних клієнтів, а для кредиту зазвичай так і є, поріг 0,25 економічно вигідніший за 0,5.

Обмеження. При порозі 0,25 позначається ризиковим ~29% клієнтів, а лише 54% з них справді стають дефолтними. Тому у практичному застосуванні такі клієнти мають не автоматично отримувати відмову, а проходити додаткову перевірку. Точний оптимальний поріг залежить від реального співвідношення вартості FN і FP, яке в нашому датасеті невідоме.


---

## Завдання 7 (Опціональне): Покращення моделі та висновки (2 бали)

**Що потрібно зробити:**

На цих же даних навчіть DecisionTreeClassifier та RandomForestClassifier. Яка з трьох моделей дає найкращий результат на тестовому наборі?

Поріг класифікації можна для порівняння якостей моделей використовувати стандартний 0.5, або той, що ви виявили, як оптимальний на попередньому кроці.
